# EIA Weekly Petroleum Inventories

This notebook downloads weekly U.S. commercial crude oil inventory data from the 
Energy Information Administration (EIA) public API. Each observation corresponds 
to the Weekly Petroleum Status Report published every Wednesday at 10:30 AM ET. 
The notebook computes the week-on-week inventory change and classifies each report 
as bearish (inventories increased) or bullish (inventories decreased), producing 
structured news events with exact timestamps. These events serve as the primary 
news signal in the baseline statistical analysis. The dataset is saved to 
`01_data/raw/macro/eia_inventories_raw.csv`.

### General imports

In [ ]:
import requests
import pandas as pd

### Download data from EIA

EIA allows to download weekly oil production and inventory reports

In [2]:
url = "https://api.eia.gov/v2/petroleum/stoc/wstk/data/"
params = {
    "api_key": "DEMO_KEY",
    "frequency": "weekly",
    "data[0]": "value",
    "facets[series][]": "WCRSTUS1",
    "start": "2020-01-01",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000
}
response = requests.get(url, params=params)
df_eia = pd.DataFrame(response.json()['response']['data'])

### Clean data

Explicit cleaning is required here because the EIA API returns a JSON response containing multiple metadata columns that are not relevant for the analysis:

- **Column selection:** The raw JSON response includes columns such as duoarea, area-name, product, process-name, series, series-description, and units. Only period (publication date) and value (inventory level in thousand barrels) are retained.

- **Type conversion:** The value column is returned as a string by the API. It is converted to numeric using pd.to_numeric(..., errors='coerce'). The errors='coerce' parameter ensures that any non-numeric values are silently converted to NaN rather than raising an exception, which prevents the pipeline from breaking on malformed entries.

- **Chronological sorting:** Records are sorted from oldest to most recent before computing weekly differences. This ordering is required to ensure that the .diff() operation produces correct sequential changes.

- **Weekly shock computation:** The week-on-week change in inventory levels is computed as inventory_change = value_this_week - value_previous_week. This delta represents the informational content of each EIA release — the market reacts not to the absolute inventory level, but to the deviation from the prior week.

- **Event direction classification:** Each weekly report is labeled according to the sign of the inventory change:

    - **bearish:** Inventories increased — signals oversupply, typically exerts downward pressure on prices \
    - **bullish:** Inventories decreased — signals scarcity, typically exerts upward pressure on prices \
    - **neutral:** No change observed

    This label serves as the structured news direction variable used in the OLS baseline analysis (*Month 1 results, p = 0.03*).
- **Exact timestamp construction:** the EIA publishes its Weekly Petroleum Status Report every Wednesday at 10:30 AM Eastern Time. An exact event timestamp is constructed by combining each report's date with this fixed publication time, then localizing it to the US/Eastern timezone. This precision is essential for correctly aligning EIA events with the hourly yfinance price data in the event window analysis.

In [3]:
df_eia = df_eia[['period', 'value']].copy()
df_eia.columns = ['date', 'inventory_mbbl']
df_eia['date'] = pd.to_datetime(df_eia['date'])
df_eia['inventory_mbbl'] = pd.to_numeric(df_eia['inventory_mbbl'], errors='coerce')
df_eia = df_eia.sort_values('date').reset_index(drop=True)

df_eia['inventory_change'] = df_eia['inventory_mbbl'].diff()
df_eia['news_direction'] = df_eia['inventory_change'].apply(
    lambda x: 'bearish' if x > 0 else ('bullish' if x < 0 else 'neutral')
)
df_eia['datetime_et'] = pd.to_datetime(df_eia['date']) + pd.Timedelta(hours=10, minutes=30)
df_eia['datetime_et'] = df_eia['datetime_et'].dt.tz_localize('US/Eastern')

print(f"Registros: {len(df_eia)}")
print(f"Rango: {df_eia['date'].min()} a {df_eia['date'].max()}")
print(f"Bearish: {(df_eia['news_direction']=='bearish').sum()}")
print(f"Bullish: {(df_eia['news_direction']=='bullish').sum()}")
print(df_eia.tail(5))


Registros: 322
Rango: 2020-01-03 00:00:00 a 2026-02-27 00:00:00
Bearish: 134
Bullish: 187
          date  inventory_mbbl  inventory_change news_direction  \
317 2026-01-30          835512           -3241.0        bullish   
318 2026-02-06          844041            8529.0        bearish   
319 2026-02-13          835256           -8785.0        bullish   
320 2026-02-20          851245           15989.0        bearish   
321 2026-02-27          854720            3475.0        bearish   

                  datetime_et  
317 2026-01-30 10:30:00-05:00  
318 2026-02-06 10:30:00-05:00  
319 2026-02-13 10:30:00-05:00  
320 2026-02-20 10:30:00-05:00  
321 2026-02-27 10:30:00-05:00  


In [4]:
filename = "eia_inventories_raw.csv"
df_eia.to_csv(f"../01_data/raw/macro/{filename}", index=False)
print(f"Guardado en 01_data/raw/macro/{filename}")

Guardado en 01_data/raw/macro/eia_inventories_raw.csv
